In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler


df = pd.read_csv("../data/preprocessed_data/forecasting_model_dataset.csv")

df["date"] = pd.to_datetime(
    df["date"]
)

df = df.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)


print(df.shape)

print(
    df["date"].min(),
    "→",
    df["date"].max()
)

display(
    df.groupby("company")[
        "date"
    ].agg(
        ["min", "max", "count"]
    )
)

(51098, 42)
2022-09-06 00:00:00 → 2026-11-12 00:00:00


,min,max,count
company,,,
99 Speed Mart,2025-07-03,2026-11-12,334
AMMB,2022-09-06,2026-11-12,1023
Alliance Bank,2022-09-06,2026-11-12,1023
Axiata,2022-09-06,2026-11-12,1022
Bursa Malaysia,2022-09-06,2026-11-12,1023
CIMB,2022-09-06,2026-11-12,1023
CelcomDigi,2022-09-06,2026-11-12,1023
Dialog Group,2022-09-06,2026-11-12,1023
Farm Fresh,2023-01-16,2026-11-12,935


In [2]:
# ============================================================
# 2. FEATURES AND TARGET
# ============================================================

feature_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",

    "daily_return",
    "intraday_return",
    "daily_range",
    "volume_change",

    "rsi_14",

    "sma_20",
    "sma_50",
    "sma_200",

    "ema_12",
    "ema_26",

    "macd",
    "macd_signal",
    "macd_histogram",

    "atr_14",

    "bollinger_width",

    "volatility_20",
    "momentum_10",

    "news_count",
    "sentiment_mean",
    "sentiment_max",
    "sentiment_min",

    "positive_score_mean",
    "negative_score_mean",
    "neutral_score_mean",

    "positive_count",
    "negative_count",
    "neutral_count"
]


target_column = "target_return_1d"


print(
    "Number of features:",
    len(feature_columns)
)


print(
    "Missing values in features:",
    df[feature_columns].isnull().sum().sum()
)


print(
    "Missing target:",
    df[target_column].isnull().sum()
)

Number of features: 32
Missing values in features: 0
Missing target: 0


In [3]:
# ============================================================
# CLEAN INVALID VALUES
# ============================================================

# Afficher les colonnes contenant +inf ou -inf
inf_counts = np.isinf(df[feature_columns]).sum()

print("Infinite values:")
print(
    inf_counts[
        inf_counts > 0
    ]
)

# Transformer +inf et -inf en NaN
df[feature_columns] = df[feature_columns].replace(
    [np.inf, -np.inf],
    np.nan
)

df[target_column] = df[target_column].replace(
    [np.inf, -np.inf],
    np.nan
)

# Supprimer uniquement les lignes devenues invalides
df = (
    df
    .dropna(
        subset=feature_columns + [target_column]
    )
    .sort_values(
        ["ticker", "date"]
    )
    .reset_index(drop=True)
)

# Vérification finale
print(
    "Remaining NaN:",
    df[
        feature_columns + [target_column]
    ].isna().sum().sum()
)

print(
    "Remaining infinity:",
    np.isinf(
        df[
            feature_columns + [target_column]
        ]
    ).sum().sum()
)

print(
    "Dataset shape after cleaning:",
    df.shape
)

Infinite values:
Series([], dtype: int64)
Remaining NaN: 0
Remaining infinity: 0
Dataset shape after cleaning: (51098, 42)


In [4]:
# ============================================================
# 3. CHRONOLOGICAL SPLIT
# ============================================================

unique_dates = np.array(
    sorted(
        df["date"].unique()
    )
)


n_dates = len(
    unique_dates
)


train_cut = int(
    n_dates * 0.70
)

val_cut = int(
    n_dates * 0.85
)


print(
    "Number of unique trading dates:",
    n_dates
)

Number of unique trading dates: 1023


In [5]:
# ------------------------------------------------------------
# One-day embargo between splits
# ------------------------------------------------------------

train_end = pd.Timestamp(
    unique_dates[
        train_cut - 2
    ]
)


val_start = pd.Timestamp(
    unique_dates[
        train_cut
    ]
)


val_end = pd.Timestamp(
    unique_dates[
        val_cut - 2
    ]
)


test_start = pd.Timestamp(
    unique_dates[
        val_cut
    ]
)


print(
    "TRAIN:",
    df["date"].min(),
    "→",
    train_end
)


print(
    "VALIDATION:",
    val_start,
    "→",
    val_end
)


print(
    "TEST:",
    test_start,
    "→",
    df["date"].max()
)

TRAIN: 2022-09-06 00:00:00 → 2025-08-07 00:00:00
VALIDATION: 2025-08-11 00:00:00 → 2026-03-27 00:00:00
TEST: 2026-03-31 00:00:00 → 2026-11-12 00:00:00


In [6]:
train_rows = df[
    df["date"] <= train_end
]

val_rows = df[
    (df["date"] >= val_start)
    &
    (df["date"] <= val_end)
]

test_rows = df[
    df["date"] >= test_start
]


print(
    "Train rows:",
    len(train_rows)
)

print(
    "Validation rows:",
    len(val_rows)
)

print(
    "Test rows:",
    len(test_rows)
)

Train rows: 35394
Validation rows: 7748
Test rows: 7854


In [7]:
print("\nTrain companies")
print(
    train_rows[
        "company"
    ].value_counts()
)


print("\nValidation companies")
print(
    val_rows[
        "company"
    ].value_counts()
)


print("\nTest companies")
print(
    test_rows[
        "company"
    ].value_counts()
)


Train companies
company
ViTrox                        715
Frontken                      715
Inari Amertron                715
Greatech                      715
AMMB                          715
CIMB                          715
RHB Bank                      715
Maybank                       715
Alliance Bank                 715
IOI Corporation               715
Bursa Malaysia                715
MISC                          715
IJM                           715
PPB Group                     715
Genting                       715
Genting Malaysia              715
Nestle Malaysia               715
YTL Corporation               715
Sime Darby                    715
Dialog Group                  715
Top Glove                     715
Hong Leong Bank               715
Petronas Dagangan             715
Telekom Malaysia              715
TIME dotCom                   715
Hartalega                     715
Petronas Chemicals            715
Sunway                        715
IHH Healthcare         

In [8]:
from sklearn.preprocessing import StandardScaler

In [9]:
# ============================================================
# 4. FEATURE SCALING
# ============================================================

scaler = StandardScaler()
scaler.fit(
    train_rows[
        feature_columns
    ]
)

StandardScaler()

In [10]:
scaled_df = df.copy()


scaled_df[
    feature_columns
] = scaler.transform(
    df[
        feature_columns
    ]
)

In [11]:
display(
    scaled_df[
        feature_columns
    ].head()
)

,open,high,low,close,volume,daily_return,intraday_return,daily_range,volume_change,rsi_14,...,news_count,sentiment_mean,sentiment_max,sentiment_min,positive_score_mean,negative_score_mean,neutral_score_mean,positive_count,negative_count,neutral_count
0,-0.015193,-0.009403,-0.011118,-0.011911,-0.517844,0.347348,0.406418,-0.088487,-0.080847,-0.592965,...,-0.16208,-0.040525,-0.040525,-0.040525,-0.121538,-0.092719,-0.096404,-0.112968,-0.084681,-0.076886
1,-0.011977,-0.014520,-0.009176,-0.011911,-0.480045,-0.029911,-0.005185,-1.036203,0.346738,-0.592965,...,-0.16208,-0.040525,-0.040525,-0.040525,-0.121538,-0.092719,-0.096404,-0.112968,-0.084681,-0.076886
2,-0.006831,-0.006205,-0.007234,-0.003547,-0.521280,0.944143,0.399068,-0.195913,-0.147096,-0.051242,...,-0.16208,-0.040525,-0.040525,-0.040525,-0.121538,-0.092719,-0.096404,-0.112968,-0.084681,-0.076886
3,-0.001684,-0.005565,-0.001407,-0.005477,-0.520270,-0.250703,-0.485019,-0.876651,-0.037288,-0.173492,...,-0.16208,-0.040525,-0.040525,-0.040525,-0.121538,-0.092719,-0.096404,-0.112968,-0.084681,-0.076886
4,-0.000398,0.002750,0.001830,0.004174,-0.524060,1.078593,0.553101,-0.211925,-0.105661,0.388458,...,-0.16208,-0.040525,-0.040525,-0.040525,-0.121538,-0.092719,-0.096404,-0.112968,-0.084681,-0.076886


In [12]:
print(
    "TRAIN means after scaling:"
)

print(
    scaled_df[
        scaled_df["date"] <= train_end
    ][
        feature_columns
    ]
    .mean()
    .head(10)
)

TRAIN means after scaling:
open              -3.212037e-17
high              -2.569630e-17
low               -3.854444e-17
close              1.927222e-17
volume            -6.424074e-18
daily_return       1.887072e-17
intraday_return   -5.621065e-18
daily_range        9.636111e-17
volume_change     -4.215799e-18
rsi_14             2.071764e-16
dtype: float64


In [13]:
print(
    "\nTRAIN standard deviations:"
)

print(
    scaled_df[
        scaled_df["date"] <= train_end
    ][
        feature_columns
    ]
    .std()
    .head(10)
)


TRAIN standard deviations:
open               1.000014
high               1.000014
low                1.000014
close              1.000014
volume             1.000014
daily_return       1.000014
intraday_return    1.000014
daily_range        1.000014
volume_change      1.000014
rsi_14             1.000014
dtype: float64


In [14]:
# ============================================================
# 5. SEQUENCE CREATION
# ============================================================

WINDOW_SIZE = 60

def create_sequences(
    data,
    feature_columns,
    target_column,
    window_size,
    train_end,
    val_start,
    val_end,
    test_start
):

    X_train = []
    y_train = []

    X_val = []
    y_val = []

    X_test = []
    y_test = []


    train_metadata = []
    val_metadata = []
    test_metadata = []


    # Important :
    # créer les séquences séparément
    # pour chaque entreprise
    for ticker, group in data.groupby(
        "ticker"
    ):

        group = (
            group
            .sort_values("date")
            .reset_index(drop=True)
        )


        features = (
            group[
                feature_columns
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        targets = (
            group[
                target_column
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        dates = (
            group[
                "date"
            ]
            .to_numpy()
        )


        companies = (
            group[
                "company"
            ]
            .to_numpy()
        )


        # --------------------------------------------
        # Sliding window
        # --------------------------------------------

        for i in range(
            window_size - 1,
            len(group)
        ):

            start = (
                i
                -
                window_size
                +
                1
            )


            X_window = features[
                start:i + 1
            ]


            y_target = targets[
                i
            ]


            end_date = pd.Timestamp(
                dates[i]
            )


            metadata = {
                "ticker":
                    ticker,

                "company":
                    companies[i],

                "date":
                    end_date
            }


            # ========================================
            # TRAIN
            # ========================================

            if end_date <= train_end:

                X_train.append(
                    X_window
                )

                y_train.append(
                    y_target
                )

                train_metadata.append(
                    metadata
                )


            # ========================================
            # VALIDATION
            # ========================================

            elif (
                end_date >= val_start
                and
                end_date <= val_end
            ):

                X_val.append(
                    X_window
                )

                y_val.append(
                    y_target
                )

                val_metadata.append(
                    metadata
                )


            # ========================================
            # TEST
            # ========================================

            elif end_date >= test_start:

                X_test.append(
                    X_window
                )

                y_test.append(
                    y_target
                )

                test_metadata.append(
                    metadata
                )


    return (
        np.array(X_train),
        np.array(y_train),

        np.array(X_val),
        np.array(y_val),

        np.array(X_test),
        np.array(y_test),

        pd.DataFrame(
            train_metadata
        ),

        pd.DataFrame(
            val_metadata
        ),

        pd.DataFrame(
            test_metadata
        )
    )

In [16]:
(
    X_train,
    y_train,

    X_val,
    y_val,

    X_test,
    y_test,

    meta_train,
    meta_val,
    meta_test

) = create_sequences(

    data=scaled_df,

    feature_columns=feature_columns,

    target_column=target_column,

    window_size=WINDOW_SIZE,

    train_end=train_end,

    val_start=val_start,

    val_end=val_end,

    test_start=test_start
)

In [17]:
print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)


print(
    "\nX_val:",
    X_val.shape
)

print(
    "y_val:",
    y_val.shape
)


print(
    "\nX_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)

X_train: (32418, 60, 32)
y_train: (32418,)

X_val: (7716, 60, 32)
y_val: (7716,)

X_test: (7854, 60, 32)
y_test: (7854,)


In [18]:
print(
    meta_train[
        "company"
    ].value_counts()
)


print(
    meta_val[
        "company"
    ].value_counts()
)


print(
    meta_test[
        "company"
    ].value_counts()
)

company
ViTrox                        656
Frontken                      656
Inari Amertron                656
Greatech                      656
AMMB                          656
CIMB                          656
Maybank                       656
RHB Bank                      656
IOI Corporation               656
Bursa Malaysia                656
IJM                           656
MISC                          656
Genting                       656
Alliance Bank                 656
Nestle Malaysia               656
YTL Corporation               656
Sime Darby                    656
PPB Group                     656
Dialog Group                  656
Tenaga Nasional               656
Hong Leong Bank               656
Petronas Dagangan             656
Genting Malaysia              656
Telekom Malaysia              656
TIME dotCom                   656
Hartalega                     656
Petronas Chemicals            656
Sunway                        656
IHH Healthcare                656
Westpo

In [19]:
# ============================================================
# 6. NAIVE BASELINE
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
baseline_predictions = np.zeros_like(
    y_test
)

In [20]:
baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)


baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)


print(
    "Baseline MAE:",
    baseline_mae
)


print(
    "Baseline RMSE:",
    baseline_rmse
)

Baseline MAE: 1.3008641004562378
Baseline RMSE: 1.9977176876116312


In [21]:
def directional_accuracy(
    y_true,
    y_pred
):

    true_direction = (
        y_true > 0
    )

    pred_direction = (
        y_pred > 0
    )

    return np.mean(
        true_direction
        ==
        pred_direction
    )

In [22]:
baseline_direction_accuracy = (
    directional_accuracy(
        y_test,
        baseline_predictions
    )
)


print(
    "Baseline Direction Accuracy:",
    baseline_direction_accuracy
)

Baseline Direction Accuracy: 0.5690094219505984


In [23]:
# ============================================================
# 7. LSTM MODEL
# ============================================================

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping


tf.random.set_seed(42)
np.random.seed(42)

In [24]:
n_features = X_train.shape[2]


lstm_model = Sequential([
    LSTM(
        64,
        input_shape=(
            WINDOW_SIZE,
            n_features
        )
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])


lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


lstm_model.summary()

c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,945 (105.25 KB)

 Trainable params: 26,945 (105.25 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [26]:
history = lstm_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        early_stopping
    ],

    verbose=1
)

Epoch 1/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 22s 19ms/step - loss: 3.3928 - mae: 1.1536 - val_loss: 3.5320 - val_mae: 1.2349
Epoch 2/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - loss: 3.3732 - mae: 1.1493 - val_loss: 3.5319 - val_mae: 1.2355
Epoch 3/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 19s 19ms/step - loss: 3.3648 - mae: 1.1481 - val_loss: 3.5343 - val_mae: 1.2361
Epoch 4/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 22s 21ms/step - loss: 3.3522 - mae: 1.1465 - val_loss: 3.5423 - val_mae: 1.2370
Epoch 5/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 22s 21ms/step - loss: 3.3400 - mae: 1.1454 - val_loss: 3.5455 - val_mae: 1.2369
Epoch 6/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 22s 21ms/step - loss: 3.3244 - mae: 1.1439 - val_loss: 3.5575 - val_mae: 1.2393
Epoch 7/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 22s 21ms/step - loss: 3.2991 - mae: 1.1422 - val_loss: 3.5607 - val_mae: 1.2400
Epoch 8/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 22s 21ms/step - loss: 3.2758 - mae: 1.1410 - val_loss: 3.5848 - val_mae: 1.2431
Epoch 9/100
1014

In [27]:
lstm_predictions = (
    lstm_model
    .predict(
        X_test
    )
    .flatten()
)

246/246 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step


In [28]:
lstm_mae = mean_absolute_error(
    y_test,
    lstm_predictions
)


lstm_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        lstm_predictions
    )
)


lstm_direction_accuracy = (
    directional_accuracy(
        y_test,
        lstm_predictions
    )
)


print(
    "LSTM MAE:",
    lstm_mae
)

print(
    "LSTM RMSE:",
    lstm_rmse
)

print(
    "LSTM Direction Accuracy:",
    lstm_direction_accuracy
)

LSTM MAE: 1.3028591871261597
LSTM RMSE: 1.9939252866026518
LSTM Direction Accuracy: 0.5192258721670486


In [29]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Zero Return",
        "LSTM"
    ],

    "MAE": [
        baseline_mae,
        lstm_mae
    ],

    "RMSE": [
        baseline_rmse,
        lstm_rmse
    ],

    "Directional Accuracy": [
        baseline_direction_accuracy,
        lstm_direction_accuracy
    ]
})


display(comparison)

,Model,MAE,RMSE,Directional Accuracy
0,Naive Zero Return,1.300864,1.997718,0.569009
1,LSTM,1.302859,1.993925,0.519226


In [31]:
# ============================================================
# 8. BiLSTM MODEL
# ============================================================

from tensorflow.keras.layers import Bidirectional

bilstm_model = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(
            WINDOW_SIZE,
            n_features
        )
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])


bilstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


bilstm_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


bilstm_history = bilstm_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        bilstm_early_stopping
    ],

    verbose=1
)

Epoch 1/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 28s 23ms/step - loss: 3.4059 - mae: 1.1576 - val_loss: 3.5361 - val_mae: 1.2360
Epoch 2/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - loss: 3.3751 - mae: 1.1498 - val_loss: 3.5386 - val_mae: 1.2357
Epoch 3/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 24s 24ms/step - loss: 3.3554 - mae: 1.1486 - val_loss: 3.5457 - val_mae: 1.2359
Epoch 4/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 22s 22ms/step - loss: 3.3326 - mae: 1.1468 - val_loss: 3.5581 - val_mae: 1.2385
Epoch 5/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - loss: 3.3090 - mae: 1.1439 - val_loss: 3.5681 - val_mae: 1.2395
Epoch 6/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 22s 21ms/step - loss: 3.2833 - mae: 1.1420 - val_loss: 3.5686 - val_mae: 1.2395
Epoch 7/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 24s 24ms/step - loss: 3.2517 - mae: 1.1387 - val_loss: 3.5734 - val_mae: 1.2427
Epoch 8/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 23s 23ms/step - loss: 3.2068 - mae: 1.1360 - val_loss: 3.6126 - val_mae: 1.2518
Epoch 9/100
1014

In [32]:
bilstm_predictions = (
    bilstm_model
    .predict(
        X_test
    )
    .flatten()
)


bilstm_mae = mean_absolute_error(
    y_test,
    bilstm_predictions
)


bilstm_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        bilstm_predictions
    )
)


bilstm_direction_accuracy = (
    directional_accuracy(
        y_test,
        bilstm_predictions
    )
)


print(
    "BiLSTM MAE:",
    bilstm_mae
)

print(
    "BiLSTM RMSE:",
    bilstm_rmse
)

print(
    "BiLSTM Direction Accuracy:",
    bilstm_direction_accuracy
)

246/246 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
BiLSTM MAE: 1.3056288957595825
BiLSTM RMSE: 1.9957516968767803
BiLSTM Direction Accuracy: 0.5183346065699007


In [33]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Zero Return",
        "LSTM",
        "BiLSTM"
    ],

    "MAE": [
        baseline_mae,
        lstm_mae,
        bilstm_mae
    ],

    "RMSE": [
        baseline_rmse,
        lstm_rmse,
        bilstm_rmse
    ],

    "Directional Accuracy": [
        baseline_direction_accuracy,
        lstm_direction_accuracy,
        bilstm_direction_accuracy
    ]
})


display(comparison)

,Model,MAE,RMSE,Directional Accuracy
0,Naive Zero Return,1.300864,1.997718,0.569009
1,LSTM,1.302859,1.993925,0.519226
2,BiLSTM,1.305629,1.995752,0.518335


In [34]:
# ============================================================
# 9. TRANSFORMER MODEL
# ============================================================

from tensorflow.keras import Model, Input

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Embedding
)


# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT_RATE = 0.2


# ------------------------------------------------------------
# Input
# ------------------------------------------------------------

inputs = Input(
    shape=(
        WINDOW_SIZE,
        n_features
    )
)


# ------------------------------------------------------------
# Project features to D_MODEL dimensions
# ------------------------------------------------------------

x = Dense(
    D_MODEL
)(inputs)


# ------------------------------------------------------------
# Positional embedding
# ------------------------------------------------------------

positions = tf.range(
    start=0,
    limit=WINDOW_SIZE,
    delta=1
)


position_embedding = Embedding(
    input_dim=WINDOW_SIZE,
    output_dim=D_MODEL
)(positions)


x = x + position_embedding


# ------------------------------------------------------------
# Multi-Head Self-Attention
# ------------------------------------------------------------

attention_output = MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=D_MODEL // NUM_HEADS
)(
    x,
    x
)


attention_output = Dropout(
    DROPOUT_RATE
)(
    attention_output
)


# Residual connection + normalization
x = LayerNormalization(
    epsilon=1e-6
)(
    x + attention_output
)


# ------------------------------------------------------------
# Feed Forward Network
# ------------------------------------------------------------

ff_output = Dense(
    FF_DIM,
    activation="relu"
)(x)


ff_output = Dropout(
    DROPOUT_RATE
)(
    ff_output
)


ff_output = Dense(
    D_MODEL
)(
    ff_output
)


# Residual connection + normalization
x = LayerNormalization(
    epsilon=1e-6
)(
    x + ff_output
)


# ------------------------------------------------------------
# Convert sequence → single representation
# ------------------------------------------------------------

x = GlobalAveragePooling1D()(
    x
)


x = Dense(
    32,
    activation="relu"
)(
    x
)


x = Dropout(
    DROPOUT_RATE
)(
    x
)


# ------------------------------------------------------------
# Regression output
# ------------------------------------------------------------

outputs = Dense(
    1
)(
    x
)


transformer_model = Model(
    inputs=inputs,
    outputs=outputs
)


transformer_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


transformer_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 60, 32)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 60, 64)    │      2,112 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 60, 64)    │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 60, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 60, 64)    │          0 │ add[0][0],        │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 60, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 60, 128)   │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 60, 64)    │      8,256 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 32)        │          0 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         33 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 37,697 (147.25 KB)

 Trainable params: 37,697 (147.25 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
transformer_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


transformer_history = transformer_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        transformer_early_stopping
    ],

    verbose=1
)

Epoch 1/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 23s 18ms/step - loss: 3.4110 - mae: 1.1547 - val_loss: 3.5321 - val_mae: 1.2323
Epoch 2/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - loss: 3.3921 - mae: 1.1488 - val_loss: 3.5311 - val_mae: 1.2341
Epoch 3/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 19s 19ms/step - loss: 3.3900 - mae: 1.1490 - val_loss: 3.5320 - val_mae: 1.2334
Epoch 4/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - loss: 3.3896 - mae: 1.1486 - val_loss: 3.5315 - val_mae: 1.2336
Epoch 5/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 17s 17ms/step - loss: 3.3903 - mae: 1.1487 - val_loss: 3.5312 - val_mae: 1.2340
Epoch 6/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - loss: 3.3906 - mae: 1.1490 - val_loss: 3.5312 - val_mae: 1.2333
Epoch 7/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 17s 17ms/step - loss: 3.3902 - mae: 1.1486 - val_loss: 3.5311 - val_mae: 1.2332
Epoch 8/100
1014/1014 ━━━━━━━━━━━━━━━━━━━━ 17s 17ms/step - loss: 3.3903 - mae: 1.1483 - val_loss: 3.5312 - val_mae: 1.2340
Epoch 9/100
1014

In [36]:
transformer_predictions = (
    transformer_model
    .predict(
        X_test
    )
    .flatten()
)


transformer_mae = mean_absolute_error(
    y_test,
    transformer_predictions
)


transformer_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        transformer_predictions
    )
)


transformer_direction_accuracy = (
    directional_accuracy(
        y_test,
        transformer_predictions
    )
)


print(
    "Transformer MAE:",
    transformer_mae
)

print(
    "Transformer RMSE:",
    transformer_rmse
)

print(
    "Transformer Direction Accuracy:",
    transformer_direction_accuracy
)

246/246 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Transformer MAE: 1.3057551383972168
Transformer RMSE: 1.9974999299126948
Transformer Direction Accuracy: 0.43099057804940155


In [37]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Zero Return",
        "LSTM",
        "BiLSTM",
        "Transformer"
    ],

    "MAE": [
        baseline_mae,
        lstm_mae,
        bilstm_mae,
        transformer_mae
    ],

    "RMSE": [
        baseline_rmse,
        lstm_rmse,
        bilstm_rmse,
        transformer_rmse
    ],

    "Directional Accuracy": [
        baseline_direction_accuracy,
        lstm_direction_accuracy,
        bilstm_direction_accuracy,
        transformer_direction_accuracy
    ]
})


display(
    comparison
)

,Model,MAE,RMSE,Directional Accuracy
0,Naive Zero Return,1.300864,1.997718,0.569009
1,LSTM,1.302859,1.993925,0.519226
2,BiLSTM,1.305629,1.995752,0.518335
3,Transformer,1.305755,1.997500,0.430991


In [ ]:
# ============================================================
# 10. TTAO IMPORT
# ============================================================

import sys
from pathlib import Path

from tensorflow.keras.optimizers import Adam


TTAO_ROOT = Path(
    r"C:\Users\bilal\Desktop\ProjetPython\ttao"
)

if str(TTAO_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(TTAO_ROOT)
    )


from ttao import (
    HyperparameterSpec,
    SearchSpace,
    compute_hpo_query_budget,
    optimize_hpo,
)


print("TTAO imported successfully.")


# ============================================================
# 11. TTAO SEARCH SPACE
# ============================================================

tta_space = SearchSpace([

    HyperparameterSpec(
        "lstm_units",
        "integer",
        32,
        128,
        step=32
    ),

    HyperparameterSpec(
        "dense_units",
        "integer",
        16,
        64,
        step=16
    ),

    HyperparameterSpec(
        "dropout",
        "continuous",
        0.1,
        0.5
    ),

    HyperparameterSpec(
        "learning_rate",
        "continuous",
        1e-4,
        5e-3,
        scale="log"
    ),

    HyperparameterSpec(
        "batch_size",
        "categorical",
        choices=(
            16,
            32,
            64
        )
    )
])


# ============================================================
# 12. TTAO OBJECTIVE FUNCTION
# ============================================================

def objective(params):

    # Libérer le modèle précédent
    tf.keras.backend.clear_session()

    # Reproductibilité
    tf.random.set_seed(42)
    np.random.seed(42)


    # Hyperparamètres proposés par TTAO
    lstm_units = int(
        params["lstm_units"]
    )

    dense_units = int(
        params["dense_units"]
    )

    dropout = float(
        params["dropout"]
    )

    learning_rate = float(
        params["learning_rate"]
    )

    batch_size = int(
        params["batch_size"]
    )


    # Construire le modèle
    model = Sequential([

        LSTM(
            lstm_units,
            input_shape=(
                WINDOW_SIZE,
                n_features
            )
        ),

        Dropout(
            dropout
        ),

        Dense(
            dense_units,
            activation="relu"
        ),

        Dense(1)
    ])


    # Compilation
    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse"
    )


    # Early stopping
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )


    # Entraînement
    model.fit(
        X_train,
        y_train,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=40,

        batch_size=batch_size,

        callbacks=[
            early_stopping
        ],

        verbose=0
    )


    # Prédiction VALIDATION uniquement
    val_predictions = (
        model
        .predict(
            X_val,
            verbose=0
        )
        .flatten()
    )


    # Fitness minimisée par TTAO
    val_rmse = np.sqrt(
        mean_squared_error(
            y_val,
            val_predictions
        )
    )


    print(
        "Params:",
        params,
        "| Validation RMSE:",
        round(
            val_rmse,
            6
        )
    )


    return float(
        val_rmse
    )




# ============================================================
# 13. TTAO OPTIMIZATION
# ============================================================

NUM_TRIANGLES = 2
MAX_ITERATIONS = 5


tta_result = optimize_hpo(

    objective_function=objective,

    search_space=tta_space,

    triangle_size_method="paper",

    alpha_method="paper",

    max_iterations=MAX_ITERATIONS,

    num_triangle=NUM_TRIANGLES,

    max_evaluations=compute_hpo_query_budget(
        NUM_TRIANGLES,
        MAX_ITERATIONS
    ),

    seed=42,

    boundary_method="reflect"
)


# ============================================================
# 14. BEST RESULT
# ============================================================

print(
    "\nBest parameters:"
)

print(
    tta_result.best_params
)


print(
    "\nBest validation RMSE:"
)

print(
    tta_result.best_fitness
)

TTAO imported successfully.



c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Params: {'lstm_units': 128, 'dense_units': 48, 'dropout': 0.13767093915505982, 'learning_rate': 0.004545201182466366, 'batch_size': 32} | Validation RMSE: 1.849688


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Params: {'lstm_units': 128, 'dense_units': 64, 'dropout': 0.25850218491202825, 'learning_rate': 0.0019222237208816639, 'batch_size': 32} | Validation RMSE: 1.849862


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


... (95 outputs left out)

c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Params: {'lstm_units': 96, 'dense_units': 48, 'dropout': 0.23673989003586218, 'learning_rate': 0.00493155847054966, 'batch_size': 32} | Validation RMSE: 1.846958


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Params: {'lstm_units': 96, 'dense_units': 48, 'dropout': 0.17981289881209356, 'learning_rate': 0.0032871184622751256, 'batch_size': 16} | Validation RMSE: 1.847222

Best parameters:
{'lstm_units': 96, 'dense_units': 48, 'dropout': 0.18192590245790197, 'learning_rate': 0.004947771920666386, 'batch_size': 32}

Best validation RMSE:
1.8451970774109967


In [37]:
# ============================================================
# 15. FINAL TTAO-LSTM
# ============================================================

best_params = tta_result.best_params

tf.keras.backend.clear_session()

tf.random.set_seed(42)
np.random.seed(42)


tta_lstm_model = Sequential([

    LSTM(
        int(best_params["lstm_units"]),
        input_shape=(
            WINDOW_SIZE,
            n_features
        )
    ),

    Dropout(
        float(best_params["dropout"])
    ),

    Dense(
        int(best_params["dense_units"]),
        activation="relu"
    ),

    Dense(1)
])


tta_lstm_model.compile(
    optimizer=Adam(
        learning_rate=float(
            best_params["learning_rate"]
        )
    ),
    loss="mse",
    metrics=["mae"]
)


tta_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


tta_history = tta_lstm_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,

    batch_size=int(
        best_params["batch_size"]
    ),

    callbacks=[
        tta_early_stopping
    ],

    verbose=1
)

Epoch 1/100


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1012/1012 ━━━━━━━━━━━━━━━━━━━━ 33s 30ms/step - loss: 3.4017 - mae: 1.1553 - val_loss: 3.4153 - val_mae: 1.2163
Epoch 2/100
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 30s 30ms/step - loss: 3.3914 - mae: 1.1491 - val_loss: 3.4171 - val_mae: 1.2187
Epoch 3/100
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 30s 30ms/step - loss: 3.3825 - mae: 1.1481 - val_loss: 3.4220 - val_mae: 1.2211
Epoch 4/100
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 30s 30ms/step - loss: 3.3830 - mae: 1.1477 - val_loss: 3.4085 - val_mae: 1.2168
Epoch 5/100
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 30s 30ms/step - loss: 3.3754 - mae: 1.1459 - val_loss: 3.4159 - val_mae: 1.2199
Epoch 6/100
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 30s 30ms/step - loss: 3.3728 - mae: 1.1463 - val_loss: 3.4137 - val_mae: 1.2191
Epoch 7/100
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 30s 30ms/step - loss: 3.3624 - mae: 1.1458 - val_loss: 3.4321 - val_mae: 1.2218
Epoch 8/100
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 30s 30ms/step - loss: 3.3511 - mae: 1.1449 - val_loss: 3.4394 - val_mae: 1.2248
Epoch 9/100
1012/1012 ━━━━━━

In [38]:
# ============================================================
# 16. FINAL TEST EVALUATION
# ============================================================

tta_predictions = (
    tta_lstm_model
    .predict(
        X_test
    )
    .flatten()
)


tta_mae = mean_absolute_error(
    y_test,
    tta_predictions
)


tta_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        tta_predictions
    )
)


tta_direction_accuracy = (
    directional_accuracy(
        y_test,
        tta_predictions
    )
)


print(
    "TTAO-LSTM MAE:",
    tta_mae
)

print(
    "TTAO-LSTM RMSE:",
    tta_rmse
)

print(
    "TTAO-LSTM Directional Accuracy:",
    tta_direction_accuracy
)

246/246 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
TTAO-LSTM MAE: 1.3084698915481567
TTAO-LSTM RMSE: 1.9988154241125944
TTAO-LSTM Directional Accuracy: 0.47937356760886174


In [39]:
comparison_final = pd.DataFrame({

    "Model": [
        "Naive Zero Return",
        "LSTM",
        "BiLSTM",
        "Transformer",
        "TTAO-LSTM"
    ],

    "MAE": [
        baseline_mae,
        lstm_mae,
        bilstm_mae,
        transformer_mae,
        tta_mae
    ],

    "RMSE": [
        baseline_rmse,
        lstm_rmse,
        bilstm_rmse,
        transformer_rmse,
        tta_rmse
    ],

    "Directional Accuracy": [
        baseline_direction_accuracy,
        lstm_direction_accuracy,
        bilstm_direction_accuracy,
        transformer_direction_accuracy,
        tta_direction_accuracy
    ]
})


display(
    comparison_final
)

,Model,MAE,RMSE,Directional Accuracy
0,Naive Zero Return,1.303613,2.003658,0.566718
1,LSTM,1.307364,1.997540,0.519862
2,BiLSTM,1.310582,2.000513,0.505730
3,Transformer,1.309454,2.003569,0.433282
4,TTAO-LSTM,1.308470,1.998815,0.479374


In [38]:
# ============================================================
# SAVE FINAL FORECAST MODEL
# ============================================================

import joblib
from pathlib import Path

forecast_dir = Path(
    "../models/forecast"
)

forecast_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Modèle LSTM retenu
lstm_model.save(
    forecast_dir / "lstm_forecast.keras"
)

# Scaler ajusté uniquement sur le train
joblib.dump(
    scaler,
    forecast_dir / "feature_scaler.pkl"
)

# Ordre exact des features
joblib.dump(
    feature_columns,
    forecast_dir / "feature_columns.pkl"
)

# Taille de fenêtre
joblib.dump(
    WINDOW_SIZE,
    forecast_dir / "window_size.pkl"
)

print("Forecast artifacts saved.")

Forecast artifacts saved.
